# Models: 2020-23, school + neighborhood

- **Paper:** Fig 1(f), Fig S1
- **Original notebook:** `45_post_both_update.ipynb`
- **Reads:**
  - `outputs/post_neighborhood_update.pkl`
  - `outputs/school_related_post_update.pkl`
- **Writes:**
  - `outputs/45_pred_actual.pkl`
  - `outputs/45_shap_values.pkl`
  - `outputs/45_*_trial.pkl`

> Note: Loading cached Optuna studies is commented out, so all four models are re-tuned on every run.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import xgboost as xgb
#from sklearn.metrics import aic
import graphviz


from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score




In [ ]:
# pip install optuna  # see requirements.txt
import optuna

# pip install shap  # see requirements.txt
import shap


In [ ]:
# set seeds

SEED = 123

random.seed(SEED)

np.random.seed(SEED)

from optuna.samplers import TPESampler
sampler = TPESampler(seed=SEED)

In [ ]:
# load data
neighborhood = pd.read_pickle(f'{OUTPUT_DIR}/post_neighborhood_update.pkl')
school_related = pd.read_pickle(f'{OUTPUT_DIR}/school_related_post_update.pkl')
tracts = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/ca_tracts_2022')

# Not used below; commented out so the 169 MB file is not needed
# vars = pd.read_csv(f'{DATA_DIR}/NHGIS/nhgis0026_csv/nhgis0026_ds249_20205_tract.csv',encoding='latin-1')


In [ ]:
# merge school to neighborhood
located_schools = gpd.GeoDataFrame(school_related,geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')
tracts = tracts.to_crs(4326)
schoolsin = gpd.sjoin(located_schools, tracts, predicate='intersects',how = 'left')

schoolsin['FIPS'] = schoolsin['STATEFP']+schoolsin['COUNTYFP']+schoolsin['TRACTCE']
#schools_with_fips = gpd.sjoin(located_schools, tracts_gdf, how="left", op="within")

columns_to_convert = ['hs_completers','college_going_count',
    'YearsTeaching', 'years_teaching_std', 'ChronicAbsenteeismEligibleCumula',
    'ChronicAbsenteeismCount', 'frpm_count', 'n_students', 'pct_passed_math',
    'pct_passed_english', 'school_expenditures', 'distance_to_csu',
    'distance_to_uc', 'AP_Count'
]

for col in columns_to_convert:
    schoolsin[col] = pd.to_numeric(schoolsin[col], errors='coerce')

tract_totals =schoolsin.groupby('FIPS').agg({
    'CountyName' :'first',


    'YearsTeaching': 'mean',
    'years_teaching_std':'mean',
    'ChronicAbsenteeismEligibleCumula':'sum',
    'ChronicAbsenteeismRate':'mean',
    'frpm_count':'sum',
    'n_students':'sum',

    'pct_passed_math':'mean',
    'college_going_count':'sum',
    'hs_completers':'sum',
    'school_expenditures':'sum',
    'distance_to_csu':'mean',
    'distance_to_uc':'mean',
    'AP_Count':'mean'
}, skipna=True).reset_index()
#tract_sum = schoolsin.groupby('TRACTCE').sum().reset_index()
tract_totals['FIPS'] = tract_totals['FIPS'].astype('float64')

schoolsin = pd.merge(tract_totals,neighborhood,on = "FIPS",how = "left")



In [ ]:
# schoolsin['enrolled_college'] = pd.to_numeric(schoolsin['enrolled_college'],errors='coerce')

# schoolsin['hs_completers'] = pd.to_numeric(schoolsin['hs_completers'],errors='coerce')

schoolsin['college_going_rate'] = 100*schoolsin['college_going_count']/schoolsin['hs_completers']
schoolsin['pct_frpm'] = schoolsin['frpm_count']/schoolsin['n_students']
# schoolsin['ChronicAbsenteeismRate'] = schoolsin['ChronicAbsenteeismCount']/schoolsin['ChronicAbsenteeismEligibleCumula']


In [ ]:
# relevant = schoolsin[['college_going_rate',
#        'YearsTeaching', 'years_teaching_std', "CountyName",
#        'ChronicAbsenteeismRate', 'pct_frpm',
#         "AP_Count", # 'pct_passed_english', 'pct_passed_math',
#        'school_expenditures','distance_to_uc',"distance_to_csu",
#                       'EP_POV150', 'EP_UNEMP','EP_NOHSDP', 'EP_UNINSUR',
#        'EP_SNGPNT', 'EP_MINRTY', 'EP_NOVEH',
#        'median_household_income', 'has_computer_internet', "english_speaking_households",
#        'employed_in_hs', 'citizen']]

relevant = schoolsin[['college_going_rate',
       'YearsTeaching', 'years_teaching_std', "CountyName",
       'ChronicAbsenteeismRate', 'pct_frpm',
         "AP_Count", # 'pct_passed_english','pct_passed_math',
       'school_expenditures','distance_to_uc',"distance_to_csu",'EP_POV150', 'EP_UNEMP', 'EP_NOHSDP', 'EP_UNINSUR',
       'EP_SNGPNT', 'EP_MINRTY', 'EP_NOVEH',
       'median_household_income', 'has_computer_internet', 'english_speaking_households',
       'employed_in_hs', 'citizen']]

#relevant  = school_merged.drop(columns = ['SchoolCode','SchoolName','CountyName','n_teachers',"Latitude",'Longitude'])

features = ['Proportion citizens (N)','Proportion high school aged employed (N)',"proportion english speaking (N)",
                        "Proportion have computer and internet (N)","median household income (N)",

                       "Proportion no vehicle(N)","proportion minority (N)",
                        "Proportion single parent (N)","proportion uninsured (N)","Proportion no hs diploma (N)",
                        "proportion unemployed (N)","Proportion below 150% poverty (N)",  'distance to nearest CSU (S)',"distance to nearest UC (S)",
                        "school expenditures (S)","number of AP classes (S)",
                        "Percent Eligible Free and Reduced Priced Meals (S)","Chronic Absenteeism Rate (S)",
                        "Years Teaching Standard Deviation (S)","Years Teaching (Mean) (S)"][::-1]


# features = ['Proportion citizens (N)','Proportion high school aged employed (N)',"Proportion english speaking (N)",
#                         "Proportion have computer and internet (N)","median household income (N)",

#                         "Proportion white (N)","Proportion no vehicle (N)","proportion minority",
#                         "Proportion single parent","proportion uninsured","Proportion no hs diploma",
#                         "proportion unemployed","Proportion below 150% poverty",
#                         'distance to nearest CSU',"distance to nearest UC",
#                         "school expenditures","number of AP classes",
#                         "Percent Eligible Free and Reduced Priced Meals","Chronic Absenteeism Rate",
#                         "Years Teaching Standard Deviation","Years Teaching (Mean)"][::-1]
#relevant  = school_merged.drop(columns = ['SchoolCode','SchoolName','CountyName','n_teachers',"Latitude",'Longitude'])


In [ ]:
# for col in relevant.columns:
#     relevant[col] = pd.to_numeric(relevant[col], errors='coerce')

cor_mat = relevant.corr(numeric_only = True)
matrix = np.triu(cor_mat)

plt.figure(figsize=(12, 8))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', fmt=".2f",mask=matrix,
            xticklabels = features,yticklabels = features)
plt.title('Correlation Matrix')
plt.show()

In [ ]:

from sklearn.model_selection import train_test_split


relevant.dropna(inplace=True)


def split_by_county(df, test_size=0.2, seed=123):
    np.random.seed(seed)
    counties = df['CountyName'].unique()
    train_list = []
    test_list = []
    small_counties = []

    for county in counties:
        county_data = df[df['CountyName'] == county]
        if len(county_data) < 5:
            small_counties.append(county_data)
        else:
            county_train, county_test = train_test_split(county_data, test_size=test_size, random_state=seed)
            train_list.append(county_train)
            test_list.append(county_test)

    if small_counties:
        small_counties_df = pd.concat(small_counties)
        small_train, small_test = train_test_split(small_counties_df, test_size=test_size, random_state=seed)
        train_list.append(small_train)
        test_list.append(small_test)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)
    return train_df, test_df

# split by counties

train_df, test_df = split_by_county(relevant)

train_df.drop(columns = "CountyName",inplace = True)
test_df.drop(columns = "CountyName",inplace = True)


train_df = train_df.apply(pd.to_numeric, errors='coerce')

test_df = test_df.apply(pd.to_numeric, errors='coerce')

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)


# x and y
X_train = train_df.drop(columns=['college_going_rate']).astype(float)
y_train = train_df['college_going_rate'].astype(float)

X_test = test_df.drop(columns=[ 'college_going_rate']).astype(float)
y_test = test_df['college_going_rate'].astype(float)

# z-score normalization
X_train_zscore = (X_train - np.mean(X_train, axis=0)) / np.std(X_train, axis=0)
X_test_zscore = (X_test - np.mean(X_test, axis=0)) / np.std(X_test, axis=0)


print('n train:', X_train_zscore.shape[0])
print('n test:', X_test_zscore.shape[0])


XGBoost hyperparameter tuner

In [ ]:
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
  # set range for hyperparams
  n_estimators = trial.suggest_int('n_estimators',10,200)
  learning_rate = trial.suggest_float('learning_rate',0.001,0.1,log=True)
  max_depth= trial.suggest_int('max_depth',1,10)
  subsample = trial.suggest_float('subsample',0.05,1)
  colsample_bytree = trial.suggest_float('colsample_bytree',0.05,1.0)
  min_child_weight = trial.suggest_int('min_child_weight',1,20)

  # define model


  model = xgb.XGBRegressor(n_estimators = n_estimators,
                                max_depth = max_depth,
                                learning_rate = learning_rate,
                                subsample = subsample,
                                colsample_bytree = colsample_bytree,
                                min_child_weight = min_child_weight)

  # model.fit(X_train, y_train)
  # y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()


  return score_r2 ,score_rmse

# if os.path.exists(f'{OUTPUT_DIR}/45_xgb_trial.pkl'):
#     study_xgb = pd.read_pickle(f'{OUTPUT_DIR}/45_xgb_trial.pkl')
# else:
study_xgb = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=100,show_progress_bar = True)
with open(f'{OUTPUT_DIR}/45_xgb_trial.pkl',"wb") as f:
     pickle.dump(study_xgb,f)
xgb_best = study_xgb.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_xgb, target_names=["r2", "rmse"])


In [ ]:
print(f"Number of trials on the Pareto front: {len(study_xgb.best_trials)}")

trial_with_highest_accuracy_xgb = max(study_xgb.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_xgb.number}")
print(f"\tparams: {trial_with_highest_accuracy_xgb.params}")
print(f"\tvalues: {trial_with_highest_accuracy_xgb.values}")

In [ ]:
# XGboost


# train
n = 100
xgb_model = xgb.XGBRegressor(**trial_with_highest_accuracy_xgb.params)
xgb_model.fit(X_train,y_train,verbose = False)


preds = xgb_model.predict(X_test)
rmse_xgb = root_mean_squared_error(y_test, preds,)
r2_xgb = r2_score(y_test, preds)
print("RMSE: ",rmse_xgb)
print("R-squared:", r2_xgb)
# show r^2
# xgb.plot_importance(model)

feature_importances_xgb = xgb_model.feature_importances_

feature_importance_dict_xgb = dict(zip(X_train.columns, feature_importances_xgb))
feature_importance_xgb = pd.DataFrame(feature_importance_dict_xgb.items(), columns=['Feature', 'Importance'])
feature_importance_xgb["Importance"] =feature_importance_xgb["Importance"]/feature_importance_xgb["Importance"].sum()

preds_xgb = preds


In [ ]:
# shap xgb
explainer_xgb = shap.Explainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test)
shap.summary_plot(shap_values_xgb, X_test,feature_names = features)

In [ ]:
# shap xgb

shap.summary_plot(shap_values_xgb, X_test,plot_type = "bar",feature_names = features)

Random Forest hyperparameter tuner

In [ ]:
from sklearn.ensemble import RandomForestRegressor


def objective_rf(trial):
  # set range for hyperparams
  n_estimators = trial.suggest_int('n_estimators',10,200)
  max_depth= trial.suggest_int('max_depth',1,10)
  min_samples_split = trial.suggest_int('min_samples_split',2,32)
  min_samples_leaf = trial.suggest_int('min_samples_leaf',2,32)

  # define model


  model = RandomForestRegressor(n_estimators = n_estimators,
                                max_depth = max_depth,
                                min_samples_split = min_samples_split,
                                min_samples_leaf = min_samples_leaf)

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
  # r2 = r2_score(y_test, y_pred)
  # rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

# if os.path.exists(f'{OUTPUT_DIR}/45_rf_trial.pkl'):
#     study_rf = pd.read_pickle(f'{OUTPUT_DIR}/45_rf_trial.pkl')
# else:
study_rf = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)

study_rf.optimize(objective_rf, n_trials=100,show_progress_bar = True)
with open(f'{OUTPUT_DIR}/45_rf_trial.pkl',"wb") as f:
    pickle.dump(study_rf,f)


rf_best = study_rf.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_rf, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_rf.best_trials)}")

trial_with_highest_accuracy_rf = max(study_rf.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_rf.number}")
print(f"\tparams: {trial_with_highest_accuracy_rf.params}")
print(f"\tvalues: {trial_with_highest_accuracy_rf.values}")

In [ ]:
rfr = RandomForestRegressor(**trial_with_highest_accuracy_rf.params)

rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
rmse_rf = root_mean_squared_error(y_test, y_pred)
r2_rf = r2_score(y_test, y_pred)
print("RMSE: ",rmse_rf)
print("R-squared:", r2_rf)
forest_importances = pd.Series(rfr.feature_importances_, index=rfr.feature_names_in_)
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)

# feature importance based on feature permutation
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rfr, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)

forest_importances = pd.Series(result.importances_mean, index=rfr.feature_names_in_)
forest_importances = forest_importances/forest_importances.sum()


preds_rf = y_pred


In [ ]:
# shap rf
explainer_rf = shap.Explainer(rfr)
shap_values_rf = explainer_rf.shap_values(X_test)
shap.summary_plot(shap_values_rf, X_test,feature_names =features)

In [ ]:
shap.summary_plot(shap_values_rf, X_test,feature_names =features,plot_type = 'bar')

Lasso hyperparameter tuner

In [ ]:
from sklearn.linear_model import Lasso

def objective_lasso(trial):
  # set range for hyperparams
  alpha = trial.suggest_float('alpha',0.02,.1)
  # define model


  model = Lasso(alpha = alpha)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
  # r2 = r2_score(y_test, y_pred)
  # rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

# if os.path.exists(f'{OUTPUT_DIR}/45_lasso_trial.pkl'):
#     study_lasso = pd.read_pickle(f'{OUTPUT_DIR}/45_lasso_trial.pkl')
# else:

study_lasso = optuna.create_study(directions = ['maximize','minimize'])

study_lasso.optimize(objective_lasso, n_trials=100,show_progress_bar = True)
with open(f'{OUTPUT_DIR}/45_lasso_trial.pkl',"wb") as f:
    pickle.dump(study_lasso,f)


lasso_best = study_lasso.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_lasso, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_lasso.best_trials)}")

trial_with_highest_accuracy_lasso = max(study_lasso.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_lasso.number}")
print(f"\tparams: {trial_with_highest_accuracy_lasso.params}")
print(f"\tvalues: {trial_with_highest_accuracy_lasso.values}")

In [ ]:

lasso = Lasso(**trial_with_highest_accuracy_lasso.params)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
rmse_lasso = root_mean_squared_error(y_test,y_pred_lasso)
r2_lasso = r2_score(y_test,y_pred_lasso)
lasso_coeff = pd.DataFrame()
lasso_coeff["Columns"] = X_train.columns
lasso_coeff['Coefficient Estimate'] = abs(pd.Series(lasso.coef_))
lasso_coeff['Coefficient Estimate'] = lasso_coeff['Coefficient Estimate']/lasso_coeff['Coefficient Estimate'].sum()

print("RMSE:",rmse_lasso)
print("R-squared:",r2_lasso)

preds_lasso = y_pred_lasso

In [ ]:
# shap lasso
explainer_lasso = shap.Explainer(lasso.predict,X_test)
shap_values_lasso = explainer_lasso.shap_values(X_test)
shap.summary_plot(shap_values_lasso, X_test)

In [ ]:
# pip install catboost  # see requirements.txt
from catboost import CatBoostRegressor

In [ ]:

def objective_cat(trial):
  # set range for hyperparams
  iterations = trial.suggest_int('iterations',10,200)
  learning_rate = trial.suggest_float('learning_rate',0.001,0.1,log=True)
  depth= trial.suggest_int('depth',1,10)
  subsample = trial.suggest_float('subsample',0.05,1)
  colsample_bylevel = trial.suggest_float('colsample_bylevel',0.05,1.0)
  min_data_in_leaf = trial.suggest_int('min_data_in_leaf',1,20)

  # define model


  model = CatBoostRegressor(iterations = iterations,
                                depth = depth,
                                learning_rate = learning_rate,
                                subsample = subsample,
                                colsample_bylevel = colsample_bylevel,
                                min_data_in_leaf = min_data_in_leaf,
                            verbose = False)

  # model.fit(X_train, y_train)
  # y_pred = model.predict(X_test)

  score_r2 = cross_val_score(model,X_train,y_train,cv = 5, scoring = "r2").mean()
  score_rmse = cross_val_score(model,X_train,y_train,cv = 5, scoring = 'neg_mean_squared_error')
  score_rmse = -1*score_rmse
  score_rmse = np.sqrt(score_rmse).mean()
#   r2 = r2_score(y_test, y_pred)
#   rmse = mean_squared_error(y_test, y_pred,squared = False)

  return score_r2 ,score_rmse

# if os.path.exists(f'{OUTPUT_DIR}/45_cat_trial.pkl'):
#     study_cat = pd.read_pickle(f'{OUTPUT_DIR}/45_cat_trial.pkl')
# else:

study_cat = optuna.create_study(directions = ['maximize','minimize'],sampler=sampler)

study_cat.optimize(objective_cat, n_trials=100,show_progress_bar = True)
with open(f'{OUTPUT_DIR}/45_cat_trial.pkl',"wb") as f:
    pickle.dump(study_cat,f)

cat_best = study_cat.best_trials

In [ ]:
optuna.visualization.plot_pareto_front(study_cat, target_names=["r2", "rmse"])

In [ ]:
print(f"Number of trials on the Pareto front: {len(study_cat.best_trials)}")

trial_with_highest_accuracy_cat = max(study_cat.best_trials, key=lambda t: t.values[1])
print(f"Trial with highest accuracy: ")
print(f"\tnumber: {trial_with_highest_accuracy_cat.number}")
print(f"\tparams: {trial_with_highest_accuracy_cat.params}")
print(f"\tvalues: {trial_with_highest_accuracy_cat.values}")

In [ ]:
catboost_model = CatBoostRegressor(**trial_with_highest_accuracy_cat.params)

catboost_model.fit(X_train, y_train, verbose= 0)

y_pred_cat = catboost_model.predict(X_test)
r2_cat = r2_score(y_test, y_pred_cat)

rmse_cat = root_mean_squared_error(y_test, y_pred_cat)
print("RMSE:", rmse_cat)
print("R-squared:",r2_cat)

feature_importance = catboost_model.get_feature_importance(type="PredictionValuesChange")

# feature importance
feature_names = X_train.columns
feature_importance_df = pd.DataFrame(list(zip(feature_names, feature_importance)),
                                      columns=['Feature', 'Importance'])
feature_importance_df['Importance'] = feature_importance_df['Importance']/feature_importance_df['Importance'].sum()
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# print(feature_importance_df)
preds_cat = y_pred_cat

In [ ]:
# shap cat


explainer_cat = shap.Explainer(catboost_model)
shap_values_cat = explainer_cat.shap_values(X_test)
shap.summary_plot(shap_values_cat, X_test,feature_names=features)

In [ ]:
shap.summary_plot(shap_values_cat, X_test,feature_names=features,plot_type = "bar")

In [ ]:
# preds = [preds_xgb,preds_rf,preds_lasso,preds_cat,y_test]
preds = pd.DataFrame({"xgb":preds_xgb,"rf":preds_rf,"lasso":preds_lasso,"cat":preds_cat,"y_test":y_test})

plt.scatter(y_test, preds.iloc[:,0], alpha=0.5, label='XGBoost',s=20)
plt.scatter(y_test, preds.iloc[:,1], alpha=0.5, label='Random Forest',s=20)
plt.scatter(y_test, preds.iloc[:,2], alpha=0.5, label='Lasso',s=20)
plt.scatter(y_test, preds.iloc[:,3], alpha=0.5, label='CatBoost',s=20)

plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', label='y = x')

plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual Values')
plt.legend()


with open(f"{OUTPUT_DIR}/45_pred_actual.pkl","wb") as fi:
    pickle.dump(preds,fi)

In [ ]:


fig, axs = plt.subplots(2,2, figsize = (12,10))
axs = axs.flatten()
axs[0].barh(feature_importance_xgb['Feature'], feature_importance_xgb['Importance'])
axs[0].set_xlabel('Importance')
axs[0].set_ylabel('Coefficient Estimate')
axs[0].set_title('XGBoost')
axs[0].set_yticklabels(features)


axs[1].barh(forest_importances.index, forest_importances.values)
axs[1].set_xlabel('Importance')
axs[1].set_ylabel('Coefficient Estimate')
axs[1].set_title('Random Forest')
axs[1].set_yticklabels([])


axs[2].barh(lasso_coeff['Columns'], lasso_coeff['Coefficient Estimate'])
axs[2].set_xlabel('Importance')
axs[2].set_ylabel('Coefficient Estimate')
axs[2].set_title('Lasso')
axs[2].set_yticklabels(features)


axs[3].barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
axs[3].set_xlabel('Importance')
axs[3].set_ylabel('Feature')
axs[3].set_title('CatBoost')
axs[3].set_yticklabels([])




plt.show()

In [ ]:
pd.DataFrame({"model":['XGB','RF','Lasso','CatBoost'],'r-squared':[r2_xgb,r2_rf,r2_lasso,r2_cat],'rmse':[rmse_xgb,rmse_rf,rmse_lasso,rmse_cat]})


In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 14))
axes = axes.flatten()


plt.sca(axes[0])

shap.summary_plot(shap_values_xgb, X_test, sort = False
                  ,show=False,plot_type = "bar",feature_names = features)
axes[0].tick_params("both",labelsize =10)
axes[0].set_xlabel("mean |Shap value|",fontsize = 10)
axes[0].set_title("XGBoost")

plt.sca(axes[1])

shap.summary_plot(shap_values_rf, X_test, sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[1].set_yticklabels([])
axes[1].tick_params("both",labelsize =10)

axes[1].set_title("Random Forest")
axes[1].set_xlabel("mean |Shap value|",fontsize = 10)
plt.sca(axes[2])

shap.summary_plot(shap_values_lasso, X_test, sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[2].set_title("Lasso")
axes[2].set_xlabel("mean |Shap value|",fontsize = 10)
axes[2].tick_params("both",labelsize =10)

plt.sca(axes[3])

shap.summary_plot(shap_values_cat, X_test,sort = False,
                  show=False,plot_type = "bar",feature_names = features)
axes[3].set_yticklabels([])
axes[3].set_title("CatBoost")
axes[3].tick_params("both",labelsize =10)

axes[3].set_xlabel("mean |Shap value|",fontsize = 10)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(30, 20))
axes = axes.flatten()


plt.sca(axes[0])

shap.summary_plot(shap_values_xgb, X_test, sort = False
                  ,show=False,feature_names = features,color_bar =False)
axes[0].tick_params("both",labelsize =10)
axes[0].set_xlabel("SHAP Value",fontsize = 10)
axes[0].set_title("XGBoost")

plt.sca(axes[1])

shap.summary_plot(shap_values_rf, X_test, sort = False,
                  show=False,feature_names = features)
axes[1].set_yticklabels([])
axes[1].tick_params("both",labelsize =10)

axes[1].set_title("Random Forest")
axes[1].set_xlabel("SHAP Value",fontsize = 10)
plt.sca(axes[2])

shap.summary_plot(shap_values_lasso, X_test, sort = False,
                  show=False,feature_names = features,color_bar =False)
axes[2].set_title("Lasso")
axes[2].set_xlabel("SHAP Value",fontsize = 10)
axes[2].tick_params("both",labelsize =10)

plt.sca(axes[3])

shap.summary_plot(shap_values_cat, X_test,sort = False,
                  show=False,feature_names = features)
axes[3].set_yticklabels([])
axes[3].set_title("CatBoost")
axes[3].tick_params("both",labelsize =10)

axes[3].set_xlabel("SHAP Value",fontsize = 10)
plt.tight_layout()
plt.show()


In [ ]:
output = [shap_values_xgb,shap_values_rf,shap_values_lasso,shap_values_cat,X_test]


with open(f'{OUTPUT_DIR}/45_shap_values.pkl','wb') as f:
  pickle.dump(output,f)